In [1]:
class process_data():
    def __init__(self):
        self
        
    def load(self, path):
        from pathlib import Path
        from langchain_core.documents import Document
        from langchain_community.document_loaders import (
            PyPDFLoader,
            TextLoader,
            Docx2txtLoader,
        )

        loader_map = {
            ".pdf": PyPDFLoader,
            ".txt": TextLoader,
            ".docx": Docx2txtLoader,
            ".md": TextLoader,
        }

        documents = []

        path = Path(path)

        for file in path.rglob("*"):
            if file.is_file():
                suffix = file.suffix.lower()

                if suffix in loader_map:
                    try:
                        loader = loader_map[suffix](str(file))
                        documents.extend(loader.load())
                        print(f"Loaded: {file.name}")
                    except Exception as e:
                        print(f"Error loading {file.name}: {e}")
                else:
                    print(f"Skipped unsupported file: {file.name}")

        self.documents = documents

        return documents
    
    def chunk(self, chunking_param = (1000, 200)):
        from langchain_text_splitters import RecursiveCharacterTextSplitter

        chunk_size, chunk_overlap = chunking_param

        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size = chunk_size,
            chunk_overlap = chunk_overlap,
            separators = ["\n\n", "\n", " ", ""]
        )    

        chunks = text_splitter.split_documents(self.documents)
        self.chunks = chunks
        print("Chunking completed")

    def create_embedding(self, model_name= "sentence-transformers/all-MiniLM-L6-v2"):
        from langchain_huggingface import HuggingFaceEmbeddings

        embeddings = HuggingFaceEmbeddings(
            model_name=model_name
        )

        self.embeddings = embeddings
        print("embedding completed")

    def store(self):
        from langchain_community.vectorstores import FAISS
        import os

        self.chunk()
        self.create_embedding()

        vectore_store = FAISS.from_documents(self.chunks, self.embeddings)

        retriver = vectore_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

        return retriver


In [ ]:
class use_llm():
    def __init__(self, retriver):
        self.retriver = retriver

    def create_prompt(self,question):
        from langchain_core.prompts import PromptTemplate

        prompt = PromptTemplate(
            template="""
            You are a RAG assistant.

            Use retrieved context first.

            If answer exists in context:
            answer from context.

            If answer is not found:
            say:
            "I could not find this information in the uploaded documents."

            Then optionally provide:
            "Based on general knowledge..."
            
            {context}
            Question: {question}
            """,
            input_variables=['context', 'question']
        )

        retirived_docs = self.retriver.invoke(question)
        context_text = "\n\n".join(doc.page_content for doc in retirived_docs)

        final_prompt = prompt.invoke({"context" : context_text, "question" : question})
        self.prompt = final_prompt

    def invoke(self, question):
        from langchain_google_genai import ChatGoogleGenerativeAI
        from dotenv import load_dotenv
        import os
        
        self.create_prompt(question)  
        
        load_dotenv()

        google_key = os.getenv("GEMINI_API_KEY")

        llm = ChatGoogleGenerativeAI(
            model="gemini-3.5-flash",
            google_api_key=google_key
        )

        print("Answer Genrating")   
        try:
            response = llm.invoke(self.prompt)
            answer = response.content

            return self.prompt, answer

        except Exception as e:
            print(f"Error while generating answer: {e}")
            return self.prompt, f"Failed to generate answer: {str(e)}"



In [3]:
data_process = process_data()

In [4]:
data_path = '../data/files'
documents = data_process.load(data_path)
chunking_param = (1000, 200)
store_path = "../data/vector_store_faiss"
retriver = data_process.store()

C:\Users\karel\AppData\Local\Temp\ipykernel_13528\3204467992.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
d:\AI & ML\Projects\AI\AI_Internship_datamites\QA_Chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded: 01_LLM_Theory_Notes.md
Loaded: 02_Pretrained_LLM_List.md
Loaded: AI Sample questions.docx
Loaded: attention.pdf
Loaded: embedding.pdf
Loaded: Harshil_Kareliya_Resume2.docx
Loaded: yolo.pdf
Chunking completed


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1630.42it/s]


embedding completed


In [29]:
llm = use_llm(retriver)

In [33]:
question = "what is transformer ?"
prompt,answer = llm.invoke(question)

Answer Genrating


In [34]:
print(prompt)

text='\n            You are a helpful assistant.\n            Answer from the provided transcript context.\n            If the context is insufficient, just say your question is not related to your data. than use your knowledge and genrate answer and say that based on my knowledge your answer is :- \n\n\n            The Transformer uses multi-head attention in three different ways:\n• In "encoder-decoder attention" layers, the queries come from the previous decoder layer,\nand the memory keys and values come from the output of the encoder. This allows every\nposition in the decoder to attend over all positions in the input sequence. This mimics the\ntypical encoder-decoder attention mechanisms in sequence-to-sequence models such as\n[38, 2, 9].\n• The encoder contains self-attention layers. In a self-attention layer all of the keys, values\nand queries come from the same place, in this case, the output of the previous layer in the\nencoder. Each position in the encoder can attend to al

In [35]:
print(answer)

your question is not related to your data. than use your knowledge and genrate answer and say that based on my knowledge your answer is :-

The Transformer is a state-of-the-art deep learning model architecture introduced in the 2017 paper *"Attention Is All You Need"* by Vaswani et al. It is primarily designed for handling sequential data, such as natural language, for tasks like translation, text generation, and summarization. 

Unlike previous sequence-to-sequence models (such as RNNs, LSTMs, or GRUs) that process data sequentially step-by-step, the Transformer processes entire sequences at once (in parallel). 

Key features and components of the Transformer include:
1. **Self-Attention Mechanism:** Allows the model to look at other words in the input sequence to help encode a specific word better, effectively capturing long-range dependencies and context regardless of distance.
2. **Multi-Head Attention:** Enables the model to jointly attend to information from different representa

: 